## 試合データと気象データのファイルに稼働率・満席率を加える
-  球場の名前が一致　⇒　観客数/スタジアムの最大観客数

In [1]:
import pandas as pd

## 2024 ~ 2026 - game_weather + stadium

In [2]:
gw24 = pd.read_csv("./games_weather_2024.csv")
gw25 = pd.read_csv("./games_weather_2025.csv")
gw26 = pd.read_csv("./games_weather_2026.csv")
stadium = pd.read_csv("./stadium.csv")

In [3]:
print(gw24.head())
print(gw25.head())
print(gw26.head())

   試合番号        年月日 曜日      球場   ビジター           ホーム    観客数   降水量    気温
0     1  2024/3/30  六  台北大巨蛋   樂天桃猿           味全龍   28618   2.5  24.0
1     2  2024/3/31  日  台北大巨蛋   中信兄弟           味全龍   20158  35.0  23.5
2     4  2024/3/31  日     台南   富邦悍將   統一7-ELEVEn獅    6219   0.0  27.3
3     5   2024/4/3  三     天母    味全龍          樂天桃猿    4088   0.0  28.6
4     7   2024/4/3  三  台北大巨蛋   台鋼雄鷹          中信兄弟   18072   0.0  28.1
   試合番号        年月日 曜日      球場          ビジター           ホーム    観客数   降水量    気温
0     1  2025/3/29  六  台北大巨蛋   統一7-ELEVEn獅          中信兄弟   40000     T  14.6
1     3  2025/3/30  日  台北大巨蛋          富邦悍將          中信兄弟   17148  22.0  13.8
2     4  2025/3/30  日     台南          台鋼雄鷹   統一7-ELEVEn獅    8177   0.0  18.0
3     5   2025/4/2  三     天母          台鋼雄鷹           味全龍    7858   0.0  15.7
4     7   2025/4/2  三     新莊          中信兄弟          富邦悍將   10819   0.0  16.0
   試合番号        年月日 曜日      球場   ビジター           ホーム    観客数  降水量    気温
0     1  2026/3/28  六  台北大巨蛋   中信兄弟          樂天桃猿

In [4]:
stadium.head()

,球場名,最大観客数
0,台南,10000
1,花蓮,5500
2,斗六,15000
3,洲際,20000
4,新莊,12150


In [5]:
# スペースを削除
gw24['球場'] = gw24['球場'].astype(str).str.strip()
gw25['球場'] = gw25['球場'].astype(str).str.strip()
gw26['球場'] = gw26['球場'].astype(str).str.strip()
stadium['球場名'] = stadium['球場名'].astype(str).str.strip()

In [17]:
# 3年分のデータを表示する
from IPython.display import display

or_df_list = []
gw_list = [gw24, gw25, gw26]
# 欠損値：降水量のTを0に変換
for df in gw_list:
    df['降水量'] = df['降水量'].astype(str).str.strip().replace(['T', 'X'], 0.0)
    df['降水量'] = pd.to_numeric(df['降水量'], errors='coerce')

for df in gw_list:
    # merge
    temp_df = pd.merge(df, stadium, left_on='球場', right_on='球場名' ,how='left', )

    # calculate occupancy rate
    temp_df['稼働率'] = temp_df['観客数']/temp_df['最大観客数']

    # 球場名を削除する
    temp_df = temp_df.copy().drop(columns='球場名')

    # リストに追加
    or_df_list.append(temp_df)

print("############################ 2024 #############################")
display(or_df_list[0])

print("############################ 2025 #############################")
display(or_df_list[1])

print("############################ 2026 #############################")
display(or_df_list[2])

############################ 2024 #############################


,試合番号,年月日,曜日,球場,ビジター,ホーム,観客数,降水量,気温,最大観客数,稼働率
0,1,2024/3/30,六,台北大巨蛋,樂天桃猿,味全龍,28618,2.5,24.0,40071,0.714182
1,2,2024/3/31,日,台北大巨蛋,中信兄弟,味全龍,20158,35.0,23.5,40071,0.503057
2,4,2024/3/31,日,台南,富邦悍將,統一7-ELEVEn獅,6219,0.0,27.3,10000,0.621900
3,5,2024/4/3,三,天母,味全龍,樂天桃猿,4088,0.0,28.6,10000,0.408800
4,7,2024/4/3,三,台北大巨蛋,台鋼雄鷹,中信兄弟,18072,0.0,28.1,40071,0.450999
...,...,...,...,...,...,...,...,...,...,...,...
355,217,2024/10/9,三,樂天桃園,中信兄弟,樂天桃猿,8853,0.0,23.0,18000,0.491833
356,218,2024/10/9,三,台南,味全龍,統一7-ELEVEn獅,5140,0.0,27.5,10000,0.514000
357,342,2024/10/10,四,澄清湖,樂天桃猿,台鋼雄鷹,8307,1.5,26.1,20000,0.415350
358,344,2024/10/10,四,新莊,統一7-ELEVEn獅,富邦悍將,7112,0.0,23.8,12150,0.585350


In [14]:
# 出力
years = [2024, 2025, 2026]

for year, df in zip(years, or_df_list):
    df.to_csv(f"occupancy_rate_{year}.csv", index=False, encoding='utf-8-sig')